# Final SKU grouping submission

This notebook loads the fine-tuned encoder, calibrates a cosine threshold on a canonical-disjoint calibration split, reports threshold sensitivity by GTIN availability, and writes exactly SKU_ID,ITEM_ID.

The final assignment is direct SKU to canonical-item assignment. It does not build connected components, so similarity chaining cannot merge unrelated products. Missing GTINs do not veto a candidate; conflicting non-empty GTINs do.

## Required inputs

Set these environment variables before running the notebook. There are no checkpoint or calibration fallbacks.

- FINETUNED_CHECKPOINT: DVC-restored SentenceTransformer checkpoint directory.
- CALIBRATION_INPUT: CSV with SKU_ID, true_item_id, and calibration_fold; each true item must occur in exactly one fold.
- SUBMISSION_PATH: optional output path; defaults to submission/SKU_ITEM_submission.csv.

In [ ]:
from pathlib import Path
import json
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import adjusted_rand_score, f1_score, precision_score, recall_score, rand_score

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError('Run this notebook from the repository or a child directory')
sys.path.insert(0, str(ROOT / 'src'))

from core.common import F, RESULTS, load_config, load_dataset_deduped
from core.attribute_conflicts import canonical_attribute_info, conflict_columns, sku_attribute_info
from core.structured_features import append_text as append_structured_text, canonical_info as canonical_structured_info, fuse_numpy, sku_info as sku_structured_info, vector as structured_vector
from pipeline import canonical_model_text, clean_sku_text, load_canonical_map, strip_schema_words

RUN_DIR = ROOT / 'training_results/20260912T211705258700Z/worker_1'
checkpoint_value = os.environ.get('FINETUNED_CHECKPOINT')
calibration_value = os.environ.get('CALIBRATION_INPUT')
if not checkpoint_value: raise RuntimeError('FINETUNED_CHECKPOINT is required; restore the checkpoint with DVC first')
if not calibration_value: raise RuntimeError('CALIBRATION_INPUT is required; provide a canonical-disjoint calibration CSV')
CHECKPOINT = Path(checkpoint_value)
CALIBRATION_INPUT = Path(calibration_value)
SUBMISSION_PATH = Path(os.environ.get('SUBMISSION_PATH', ROOT / 'submission/SKU_ITEM_submission.csv'))
THRESHOLDS = np.round(np.arange(0.80, 0.981, 0.01), 2)
if not CHECKPOINT or not CHECKPOINT.exists():
    raise FileNotFoundError('FINETUNED_CHECKPOINT must point to a DVC-restored checkpoint directory')
if not CALIBRATION_INPUT or not CALIBRATION_INPUT.exists():
    raise FileNotFoundError('CALIBRATION_INPUT must point to the canonical-disjoint calibration CSV')
CFG = load_config()
SF_CFG = CFG['training']['structured_features']
print({'checkpoint': str(CHECKPOINT), 'calibration': str(CALIBRATION_INPUT), 'run_dir': str(RUN_DIR)})

In [ ]:
canonical = load_canonical_map()
item_ids = [str(x) for x in canonical]
records = pd.read_csv(RESULTS / F['canonical_records'], dtype=str, keep_default_na=False)
record_map = {str(row['gtin']): row.to_dict() for _, row in records.iterrows()}
model = SentenceTransformer(str(CHECKPOINT))
sf_enabled = bool(SF_CFG['enabled'])
sf_text = sf_enabled and bool(SF_CFG['append_to_text'])
sf_weight = float(SF_CFG['embedding_weight']) if sf_enabled and bool(SF_CFG['feed_to_loss']) else 0.0

def _text_and_info(df):
    sku_infos = [sku_structured_info(row.get('title', ''), row.get('attributes', row.get('attr', ''))) if sf_enabled else {'volume': set(), 'pack': set()} for _, row in df.iterrows()]
    texts = [append_structured_text(strip_schema_words(clean_sku_text(row.get('title', ''), row.get('attributes', row.get('attr', '')), row.get('brand', ''))), info, enabled=sf_text) for (_, row), info in zip(df.iterrows(), sku_infos)]
    return texts, sku_infos

item_infos = [canonical_structured_info(record_map.get(item_id, {})) if sf_enabled else {'volume': set(), 'pack': set()} for item_id in item_ids]
item_texts = [append_structured_text(strip_schema_words(canonical_model_text(canonical[item_id])), info, enabled=sf_text) for item_id, info in zip(item_ids, item_infos)]
item_emb = model.encode(item_texts, batch_size=128, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
item_features = np.asarray([structured_vector(info, volume_scale_ml=float(SF_CFG['volume_scale_ml']), pack_scale=float(SF_CFG['pack_scale']), max_set_size=int(SF_CFG['max_set_size'])) for info in item_infos], dtype=np.float32)
item_emb = fuse_numpy(item_emb, item_features, sf_weight)
item_emb_t = torch.as_tensor(item_emb)
print(f'loaded {len(item_ids):,} canonical items from {CHECKPOINT}')

In [ ]:
def _gtin(value):
    text = str(value or '').strip()
    return '' if text.lower() in {'nan', 'none', 'null'} else text

def gtin_status(sku_gtin, candidate_gtin):
    a, b = _gtin(sku_gtin), _gtin(candidate_gtin)
    if not a and not b: return 'both_missing'
    if not a or not b: return 'one_missing'
    return 'both_equal' if a == b else 'different'

def score_candidates(skus, top_k=20):
    skus = skus.copy()
    if 'SKU_ID' not in skus and 'product_id' in skus: skus = skus.rename(columns={'product_id': 'SKU_ID'})
    if 'SKU_ID' not in skus: raise ValueError('input must contain SKU_ID or product_id')
    skus['SKU_ID'] = skus['SKU_ID'].astype(str)
    texts, sku_infos = _text_and_info(skus)
    emb = model.encode(texts, batch_size=128, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
    features = np.asarray([structured_vector(info, volume_scale_ml=float(SF_CFG['volume_scale_ml']), pack_scale=float(SF_CFG['pack_scale']), max_set_size=int(SF_CFG['max_set_size'])) for info in sku_infos], dtype=np.float32)
    emb = fuse_numpy(emb, features, sf_weight)
    hits = util.semantic_search(torch.as_tensor(emb), item_emb_t, top_k=min(top_k, len(item_ids)))
    rows = []
    for pos, (_, row) in enumerate(skus.iterrows()):
        sku_id = str(row['SKU_ID'])
        sku_gtin = _gtin(row.get('barcode', row.get('gtin', '')))
        candidate_indexes = {int(hit['corpus_id']) for hit in hits[pos]}
        if sku_gtin in item_ids: candidate_indexes.add(item_ids.index(sku_gtin))
        left = sku_attribute_info(row.get('title', ''), row.get('attributes', row.get('attr', '')))
        for idx in candidate_indexes:
            candidate_gtin = item_ids[idx]
            right = canonical_attribute_info(record_map.get(candidate_gtin, {}))
            rules = conflict_columns(left, right)
            score = float(np.dot(emb[pos], item_emb[idx]))
            rows.append({'SKU_ID': sku_id, 'sku_gtin': sku_gtin, 'candidate_gtin': candidate_gtin, 'score': score, 'exact_gtin': int(bool(sku_gtin) and sku_gtin == candidate_gtin), 'gtin_status': gtin_status(sku_gtin, candidate_gtin), 'rule_ok': int(rules['attribute_conflict_type'] == 'none'), 'attribute_matches': int(sum(not rules[k] for k in ('volume_conflict', 'pack_conflict', 'flavor_conflict')))})
    return pd.DataFrame(rows)

In [ ]:
def choose_assignments(candidates, threshold):
    if candidates.empty: return pd.DataFrame(columns=['SKU_ID', 'ITEM_ID', 'score', 'gtin_status'])
    c = candidates.copy()
    c['gtin_compatible'] = c['gtin_status'].ne('different')
    c['accepted'] = c['gtin_compatible'] & (c['exact_gtin'].astype(bool) | (c['rule_ok'].astype(bool) & (c['score'] >= float(threshold))))
    c = c[c['accepted']].sort_values(['SKU_ID', 'exact_gtin', 'score', 'attribute_matches', 'candidate_gtin'], ascending=[True, False, False, False, True], kind='mergesort')
    best = c.drop_duplicates('SKU_ID', keep='first')[['SKU_ID', 'candidate_gtin', 'score', 'gtin_status']].rename(columns={'candidate_gtin': 'ITEM_ID'})
    all_skus = candidates[['SKU_ID']].drop_duplicates()
    out = all_skus.merge(best, on='SKU_ID', how='left')
    out['ITEM_ID'] = out['ITEM_ID'].fillna('UNMATCHED_' + out['SKU_ID'].astype(str))
    return out

def prediction_metrics(pred, truth):
    x = truth[['SKU_ID', 'true_item_id']].merge(pred[['SKU_ID', 'ITEM_ID']], on='SKU_ID', how='inner')
    if x.empty: raise ValueError('no calibration rows matched predictions')
    return {'n': int(len(x)), 'rand_index': float(rand_score(x.true_item_id, x.ITEM_ID)), 'adjusted_rand': float(adjusted_rand_score(x.true_item_id, x.ITEM_ID)), 'precision': float(precision_score(x.true_item_id, x.ITEM_ID, average='micro')), 'recall': float(recall_score(x.true_item_id, x.ITEM_ID, average='micro')), 'f1': float(f1_score(x.true_item_id, x.ITEM_ID, average='micro'))}

def gtin_metrics(pred, truth, fold, threshold):
    x = truth[['SKU_ID', 'true_item_id']].merge(pred[['SKU_ID', 'ITEM_ID', 'gtin_status']], on='SKU_ID', how='inner')
    rows = []
    groups = list(x.groupby('gtin_status', dropna=False)) + [('ALL', x)]
    for status, g in groups:
        m = prediction_metrics(g[['SKU_ID', 'ITEM_ID']], g[['SKU_ID', 'true_item_id']])
        rows.append({'check_fold': fold, 'threshold': float(threshold), 'gtin_status': status, **m})
    return rows

## Canonical-disjoint threshold calibration

Each calibration fold owns whole true items, not random SKU rows. The threshold is selected on the non-held-out folds and measured on the held-out fold. Ties favor the highest threshold, which is conservative against over-merging. The final threshold is the median of fold-selected thresholds.

In [ ]:
labels = pd.read_csv(CALIBRATION_INPUT, dtype=str, keep_default_na=False)
required = {'SKU_ID', 'true_item_id', 'calibration_fold'}
missing = required - set(labels.columns)
if missing: raise ValueError(f'calibration input missing columns: {sorted(missing)}')
base = load_dataset_deduped().rename(columns={'product_id': 'SKU_ID'})
cal = base.merge(labels[['SKU_ID', 'true_item_id', 'calibration_fold']], on='SKU_ID', how='inner')
cal['SKU_ID'] = cal['SKU_ID'].astype(str)
cal['true_item_id'] = cal['true_item_id'].astype(str)
folds_per_item = cal.groupby('true_item_id')['calibration_fold'].nunique()
if (folds_per_item > 1).any(): raise ValueError('calibration is not canonical-disjoint: an item appears in multiple folds')
cal_candidates = score_candidates(cal)
cal_truth = cal[['SKU_ID', 'true_item_id', 'calibration_fold']].drop_duplicates('SKU_ID')
folds = sorted(cal['calibration_fold'].unique())
selected = []
sensitivity = []
for fold in folds:
    fit_skus = set(cal.loc[cal.calibration_fold != fold, 'SKU_ID'])
    check_skus = set(cal.loc[cal.calibration_fold == fold, 'SKU_ID'])
    fit_truth = cal_truth[cal_truth.SKU_ID.isin(fit_skus)]
    check_truth = cal_truth[cal_truth.SKU_ID.isin(check_skus)]
    fit_candidates = cal_candidates[cal_candidates.SKU_ID.isin(fit_skus)]
    check_candidates = cal_candidates[cal_candidates.SKU_ID.isin(check_skus)]
    fit_rows = []
    for t in THRESHOLDS:
        fit_pred = choose_assignments(fit_candidates, t)
        fit_rows.append((t, prediction_metrics(fit_pred, fit_truth)['rand_index']))
    best_t = max(fit_rows, key=lambda x: (x[1], x[0]))[0]
    selected.append({'check_fold': fold, 'selected_threshold': float(best_t), 'fit_rand_index': float(dict(fit_rows)[best_t])})
    for t in THRESHOLDS:
        sensitivity.extend(gtin_metrics(choose_assignments(check_candidates, t), check_truth, fold, t))
selected_df = pd.DataFrame(selected)
FINAL_THRESHOLD = float(selected_df.selected_threshold.median())
sensitivity_df = pd.DataFrame(sensitivity)
OUT_DIR = SUBMISSION_PATH.parent
OUT_DIR.mkdir(parents=True, exist_ok=True)
selected_df.to_csv(OUT_DIR / 'threshold_selection_by_fold.csv', index=False)
sensitivity_df.to_csv(OUT_DIR / 'threshold_sensitivity_by_gtin_status.csv', index=False)
print({'folds': folds, 'selected_thresholds': selected_df.selected_threshold.tolist(), 'final_threshold': FINAL_THRESHOLD})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for status, g in sensitivity_df.groupby('gtin_status'):
    if status == 'ALL': continue
    curve = g.groupby('threshold', as_index=False).rand_index.mean()
    ax.plot(curve.threshold, curve.rand_index, marker='o', label=status)
ax.axvline(FINAL_THRESHOLD, color='black', linestyle='--', label=f'final={FINAL_THRESHOLD:.2f}')
ax.set(xlabel='Cosine threshold', ylabel='Rand Index', title='Threshold sensitivity by GTIN availability')
ax.legend()
fig.tight_layout()
plot_path = OUT_DIR / 'threshold_sensitivity_by_gtin_status.png'
fig.savefig(plot_path, dpi=160)
plt.show()
print(plot_path)

## Generate the final submission

An unmatched SKU receives UNMATCHED_<SKU_ID>, creating a unique singleton. Equal non-empty GTINs are an exact-match priority; a non-empty conflicting GTIN is never assigned. One-missing and both-missing cases are scored normally. Tie-breaking is exact GTIN, cosine score, number of matching attributes, then lexicographically smallest canonical GTIN.

In [ ]:
final_skus = load_dataset_deduped()
final_candidates = score_candidates(final_skus)
final_pred = choose_assignments(final_candidates, FINAL_THRESHOLD)
submission = final_pred[['SKU_ID', 'ITEM_ID']].copy()
if list(submission.columns) != ['SKU_ID', 'ITEM_ID']: raise AssertionError('submission schema is not exactly SKU_ID,ITEM_ID')
if submission.SKU_ID.duplicated().any(): raise AssertionError('submission has duplicate SKU_ID values')
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False)
diagnostics = final_pred.merge(final_candidates.groupby('SKU_ID', as_index=False).agg(n_candidates=('candidate_gtin', 'nunique')), on='SKU_ID', how='left')
diagnostics.to_csv(SUBMISSION_PATH.with_name(SUBMISSION_PATH.stem + '_diagnostics.csv'), index=False)
with open(SUBMISSION_PATH.with_name('threshold_selection.json'), 'w', encoding='utf-8') as handle:
    json.dump({'method': 'canonical disjoint fold calibration', 'final_threshold': FINAL_THRESHOLD, 'tie_break': ['exact_gtin', 'score', 'attribute_matches', 'candidate_gtin'], 'unmatched_item_id': 'UNMATCHED_<SKU_ID>', 'no_transitive_chaining': True}, handle, indent=2)
print(submission.head())
print({'rows': len(submission), 'unique_items': submission.ITEM_ID.nunique(), 'unmatched': int(submission.ITEM_ID.str.startswith('UNMATCHED_').sum()), 'path': str(SUBMISSION_PATH)})